# Validation: N_SAMPLES=10, N_LOCAL=10

This notebook validates LLM responses from the enforce_knowledge experiment with config:
- **N_SAMPLES:** 10 training + 10 prediction examples
- **N_LOCAL:** 10 SHAP + LIME local explanations

## Validation goals
1. **Feature ranking accuracy** - Compare LLM's top-3 features per class against ground truth SHAP rankings
2. **SHAP citation accuracy** - Check if LLM cites SHAP values correctly (Phase 2)
3. **Fabrication detection** - Flag invented percentages, misclassifications, overvaluation of User_Agent
4. **Phase comparison** - Does Phase 2 (with XAI) improve over Phase 1 (raw data)?

In [1]:
import json
import pandas as pd
from IPython.display import display, Markdown, HTML
import sys
sys.path.append('.')
from validation_helpers import (
    compute_ground_truth,
    extract_all_rankings,
    extract_shap_values,
    score_shap_citations,
    check_user_agent_overvaluation,
    check_fabricated_percentages,
    ranking_accuracy,
    ranking_top3_set_overlap,
    display_response,
    display_ranking_comparison
)

In [2]:
# Load results
with open('resultados_enforce_knowledge_samples_10_local_10.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded results for {len(results)} models")
print(f"Models: {list(results.keys())}")

Loaded results for 4 models
Models: ['glm-4.7-flash:latest', 'qwen3:14b', 'gpt-oss:20b', 'qwen3:30b']


In [3]:
# Compute ground truth
gt = compute_ground_truth('../Network_logs.csv')

print("=== Ground Truth SHAP Rankings (Top-3 per class) ===")
for cls_name, ranking in gt['shap_rankings'].items():
    print(f"{cls_name}: {' > '.join(ranking[:3])}")

print(f"\nModel accuracy: {gt['accuracy']:.4f}")
print(f"Features: {gt['feature_names']}")
print(f"Classes: {gt['class_names']}")

=== Ground Truth SHAP Rankings (Top-3 per class) ===
BotAttack: Port > Status > Payload_Size
Normal: Port > Status > Payload_Size
PortScan: Payload_Size > Status > Port

Model accuracy: 0.9970
Features: ['Port', 'Request_Type', 'Protocol', 'Payload_Size', 'User_Agent', 'Status']
Classes: ['BotAttack', 'Normal', 'PortScan']


### Ground Truth Reference

**Expected top-3 feature importance per class:**
- **BotAttack:** Port > Status > Payload_Size
- **Normal:** Port > Status > Payload_Size  
- **PortScan:** Payload_Size > Status > Port

**Key insight:** User_Agent has near-zero SHAP importance (< 0.006) despite being commonly overvalued by LLMs.

In [4]:
# Validation results storage
validation_results = {}

for model_name, model_data in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)
    
    validation_results[model_name] = {}
    
    for phase_key, phase_label in [
        ('phase1_without_xai', 'Phase 1 (No XAI)'),
        ('phase2_with_xai', 'Phase 2 (With XAI)')
    ]:
        if phase_key not in model_data:
            print(f"  {phase_label}: SKIPPED (not present)")
            continue
        
        text = model_data[phase_key]
        print(f"\n  {phase_label}: {len(text)} chars")
        
        # Extract rankings
        predicted_rankings = extract_all_rankings(text, gt['class_names'])
        
        # Ranking accuracy (position match)
        correct, total, per_class = ranking_accuracy(predicted_rankings, gt['shap_rankings'], gt['class_names'])
        ranking_acc = correct / total if total > 0 else 0
        
        # Set overlap (order-independent)
        overlap_correct, overlap_total = ranking_top3_set_overlap(predicted_rankings, gt['shap_rankings'], gt['class_names'])
        set_overlap = overlap_correct / overlap_total if overlap_total > 0 else 0
        
        # User_Agent overvaluation check
        ua_hits = check_user_agent_overvaluation(text)
        
        # Fabricated percentages (Phase 1 only - no SHAP to back them up)
        if phase_key == 'phase1_without_xai':
            fab_pct = check_fabricated_percentages(text)
        else:
            fab_pct = []
        
        # SHAP citation accuracy (Phase 2 only)
        if phase_key == 'phase2_with_xai':
            cited_shap = extract_shap_values(text, gt['class_names'])
            shap_checks = score_shap_citations(cited_shap, gt['shap_ground_truth'], gt['class_names'])
            n_exact = sum(1 for c in shap_checks if c['status'] == 'EXACT')
            n_close = sum(1 for c in shap_checks if c['status'] == 'CLOSE')
            n_off = sum(1 for c in shap_checks if c['status'] == 'OFF')
        else:
            cited_shap = {}
            shap_checks = []
            n_exact = n_close = n_off = 0
        
        validation_results[model_name][phase_key] = {
            'text_length': len(text),
            'predicted_rankings': predicted_rankings,
            'ranking_accuracy': ranking_acc,
            'set_overlap': set_overlap,
            'per_class': per_class,
            'user_agent_overvaluation': ua_hits,
            'fabricated_percentages': fab_pct,
            'shap_citations': cited_shap if phase_key == 'phase2_with_xai' else None,
            'shap_check_counts': {'exact': n_exact, 'close': n_close, 'off': n_off} if phase_key == 'phase2_with_xai' else None,
        }
        
        print(f"    Ranking accuracy (position match): {ranking_acc:.1%} ({correct}/{total})")
        print(f"    Set overlap (order-independent): {set_overlap:.1%}")
        print(f"    User_Agent overvaluation hits: {len(ua_hits)}")
        if phase_key == 'phase1_without_xai':
            print(f"    Fabricated percentages: {len(fab_pct)}")
        else:
            print(f"    SHAP citations: {n_exact} exact, {n_close} close, {n_off} off")


Model: glm-4.7-flash:latest

  Phase 1 (No XAI): 6332 chars
    Ranking accuracy (position match): 0.0% (0/9)
    Set overlap (order-independent): 0.0%
    User_Agent overvaluation hits: 2
    Fabricated percentages: 0

  Phase 2 (With XAI): 4974 chars
    Ranking accuracy (position match): 0.0% (0/9)
    Set overlap (order-independent): 33.3%
    User_Agent overvaluation hits: 0
    SHAP citations: 0 exact, 0 close, 0 off

Model: qwen3:14b

  Phase 1 (No XAI): 6113 chars
    Ranking accuracy (position match): 11.1% (1/9)
    Set overlap (order-independent): 66.7%
    User_Agent overvaluation hits: 1
    Fabricated percentages: 0

  Phase 2 (With XAI): 5939 chars
    Ranking accuracy (position match): 0.0% (0/9)
    Set overlap (order-independent): 0.0%
    User_Agent overvaluation hits: 0
    SHAP citations: 0 exact, 2 close, 7 off

Model: gpt-oss:20b

  Phase 1 (No XAI): 11157 chars
    Ranking accuracy (position match): 11.1% (1/9)
    Set overlap (order-independent): 33.3%
    Use

## Summary Table

In [5]:
# Build summary DataFrame
summary_rows = []

for model_name, phases in validation_results.items():
    for phase_key, metrics in phases.items():
        summary_rows.append({
            'model': model_name,
            'phase': 'Phase 1 (No XAI)' if phase_key == 'phase1_without_xai' else 'Phase 2 (With XAI)',
            'ranking_accuracy': metrics['ranking_accuracy'],
            'set_overlap': metrics['set_overlap'],
            'user_agent_overvaluation': len(metrics['user_agent_overvaluation']),
            'fabricated_percentages': len(metrics['fabricated_percentages']) if phase_key == 'phase1_without_xai' else 0,
            'shap_exact': metrics['shap_check_counts']['exact'] if phase_key == 'phase2_with_xai' else None,
            'shap_close': metrics['shap_check_counts']['close'] if phase_key == 'phase2_with_xai' else None,
            'shap_off': metrics['shap_check_counts']['off'] if phase_key == 'phase2_with_xai' else None,
            'text_length': metrics['text_length'],
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,model,phase,ranking_accuracy,set_overlap,user_agent_overvaluation,fabricated_percentages,shap_exact,shap_close,shap_off,text_length
0,glm-4.7-flash:latest,Phase 1 (No XAI),0.000000,0.000000,2,0,NaN,NaN,NaN,6332
1,glm-4.7-flash:latest,Phase 2 (With XAI),0.000000,0.333333,0,0,0.0,0.0,0.0,4974
2,qwen3:14b,Phase 1 (No XAI),0.111111,0.666667,1,0,NaN,NaN,NaN,6113
3,qwen3:14b,Phase 2 (With XAI),0.000000,0.000000,0,0,0.0,2.0,7.0,5939
4,gpt-oss:20b,Phase 1 (No XAI),0.111111,0.333333,0,0,NaN,NaN,NaN,11157
5,gpt-oss:20b,Phase 2 (With XAI),0.777778,1.000000,0,0,0.0,0.0,0.0,8029
6,qwen3:30b,Phase 1 (No XAI),0.000000,0.000000,0,0,NaN,NaN,NaN,6075
7,qwen3:30b,Phase 2 (With XAI),0.000000,0.000000,0,0,0.0,0.0,0.0,0


## Ranking Comparison Tables

In [6]:
# Display ranking comparison for each model/phase
for model_name, phases in validation_results.items():
    print(f"\n{'='*60}")
    print(f"{model_name}")
    print('='*60)
    
    for phase_key, metrics in phases.items():
        phase_label = 'Phase 1 (No XAI)' if phase_key == 'phase1_without_xai' else 'Phase 2 (With XAI)'
        print(f"\n--- {phase_label} ---")
        display_ranking_comparison(
            model_name, phase_label,
            metrics['predicted_rankings'], gt['shap_rankings'], gt['class_names']
        )


glm-4.7-flash:latest

--- Phase 1 (No XAI) ---


,Ground Truth,Phase 1 (No XAI) Ranking,Position Matches
Class,,,
BotAttack,Port > Status > Payload_Size,? > ? > ?,0/3
Normal,Port > Status > Payload_Size,? > ? > ?,0/3
PortScan,Payload_Size > Status > Port,? > ? > ?,0/3



--- Phase 2 (With XAI) ---


,Ground Truth,Phase 2 (With XAI) Ranking,Position Matches
Class,,,
BotAttack,Port > Status > Payload_Size,Status > Protocol > ?,0/3
Normal,Port > Status > Payload_Size,Status > Protocol > ?,0/3
PortScan,Payload_Size > Status > Port,Status > Protocol > ?,0/3



qwen3:14b

--- Phase 1 (No XAI) ---


...

,Ground Truth,Phase 2 (With XAI) Ranking,Position Matches
Class,,,
BotAttack,Port > Status > Payload_Size,Port > Status > Payload_Size,3/3
Normal,Port > Status > Payload_Size,Port > Status > Payload_Size,3/3
PortScan,Payload_Size > Status > Port,Port > Status > Payload_Size,1/3



qwen3:30b

--- Phase 1 (No XAI) ---


,Ground Truth,Phase 1 (No XAI) Ranking,Position Matches
Class,,,
BotAttack,Port > Status > Payload_Size,? > ? > ?,0/3
Normal,Port > Status > Payload_Size,? > ? > ?,0/3
PortScan,Payload_Size > Status > Port,? > ? > ?,0/3



--- Phase 2 (With XAI) ---


,Ground Truth,Phase 2 (With XAI) Ranking,Position Matches
Class,,,
BotAttack,Port > Status > Payload_Size,? > ? > ?,0/3
Normal,Port > Status > Payload_Size,? > ? > ?,0/3
PortScan,Payload_Size > Status > Port,? > ? > ?,0/3


## Detailed Inspection

In [7]:
# Show flagged issues
for model_name, phases in validation_results.items():
    for phase_key, metrics in phases.items():
        phase_label = 'Phase 1' if phase_key == 'phase1_without_xai' else 'Phase 2'
        
        if metrics['user_agent_overvaluation']:
            print(f"\n{model_name} - {phase_label}: User_Agent overvaluation ({len(metrics['user_agent_overvaluation'])} hits)")
            for hit in metrics['user_agent_overvaluation'][:3]:
                print(f"  - {hit}")
        
        if metrics['fabricated_percentages']:
            print(f"\n{model_name} - {phase_label}: Fabricated percentages ({len(metrics['fabricated_percentages'])} hits)")
            for item in metrics['fabricated_percentages'][:3]:
                print(f"  - {item['feature']}: {item['percentage']}%")
        
        if phase_key == 'phase2_with_xai' and metrics['shap_check_counts']['off'] > 0:
            print(f"\n{model_name} - {phase_label}: SHAP citations off ({metrics['shap_check_counts']['off']} values)")


glm-4.7-flash:latest - Phase 1: User_Agent overvaluation (2 hits)
  - ior:** The model learns that `User_Agent >= 4` (Tools) is a strong indicator of automated scanning or attack behavior, whereas agencies starting at `0` (Standard Browsers) are predominantly classified as Normal or ben
  - (4444) and 11 (31337).
*   **User Agent (High Impact):**
    *   **Analysis:** Thi

qwen3:14b - Phase 1: User_Agent overvaluation (1 hits)
  - ybersecurity relevance):**  

1. **User_Agent (Categorical, Encoded)**

qwen3:14b - Phase 2: SHAP citations off (7 values)


## Full Responses (for manual inspection)

In [8]:
# Optionally display full responses (commented out by default - very verbose)
# Uncomment to inspect specific model/phase

# model_to_inspect = "qwen3:14b"
# phase_to_inspect = "phase2_with_xai"
# display_response(
#     model_to_inspect, 
#     phase_to_inspect.replace('_without_xai', 'Without XAI').replace('_with_xai', 'With XAI'),
#     results[model_to_inspect][phase_to_inspect]
# )